# 🏜️ Off-Road Desert Semantic Segmentation — Google Colab

**Train DeepLabV3+ (ResNet101) on a free GPU in Google Colab**

This notebook guides you through the complete training pipeline:
1. ✅ Mount Google Drive & upload data
2. ✅ Install dependencies
3. ✅ Verify GPU is available
4. ✅ Train the model (with full ResNet101 backbone)
5. ✅ Evaluate & visualize results
6. ✅ Download trained model to your Mac

---

## 📋 Prerequisites

Before starting, run this on your Mac terminal:
```bash
cd /Users/abhimanraj/Downloads/Offroad_Segmentation_Scripts/ENV_SETUP
bash prepare_for_colab.sh
```

Then upload these two files to **Google Drive** (root or a folder):
- `offroad_scripts.zip`
- `offroad_dataset.zip`

---
## 🔧 Step 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

---
## 📁 Step 2: Setup Project Directory

**⚠️ IMPORTANT:** Update `DRIVE_FOLDER` below if you uploaded the zip files to a subfolder in Google Drive.

In [ ]:
import os

# ========================================
# UPDATE THIS if your zips are in a subfolder
# e.g., "/content/drive/MyDrive/ML_Projects"
# ========================================
DRIVE_FOLDER = "/content/drive/MyDrive"

# Project directory (inside Colab runtime - fast SSD)
PROJECT_DIR = "/content/offroad_segmentation"
os.makedirs(PROJECT_DIR, exist_ok=True)

# Verify zip files exist
scripts_zip = os.path.join(DRIVE_FOLDER, "offroad_scripts.zip")
dataset_zip = os.path.join(DRIVE_FOLDER, "offroad_dataset.zip")

assert os.path.exists(scripts_zip), f"❌ offroad_scripts.zip not found at {scripts_zip}"
assert os.path.exists(dataset_zip), f"❌ offroad_dataset.zip not found at {dataset_zip}"
print("✅ Both zip files found on Google Drive!")

In [ ]:
# Unzip scripts into the project directory
!unzip -o "{scripts_zip}" -d "{PROJECT_DIR}"
print("\n✅ Scripts extracted!")

# Unzip dataset into the project directory
!mkdir -p "{PROJECT_DIR}/Offroad_Segmentation_Training_Dataset"
!unzip -o "{dataset_zip}" -d "{PROJECT_DIR}/Offroad_Segmentation_Training_Dataset"
print("\n✅ Dataset extracted!")

# Create output directories
!mkdir -p "{PROJECT_DIR}/checkpoints" "{PROJECT_DIR}/outputs" "{PROJECT_DIR}/predictions"

# Verify structure
print("\n📁 Project structure:")
!ls -la "{PROJECT_DIR}/"
print("\n📁 Dataset:")
!echo "Train images: $(ls {PROJECT_DIR}/Offroad_Segmentation_Training_Dataset/train/Color_Images/ | wc -l)"
!echo "Train masks:  $(ls {PROJECT_DIR}/Offroad_Segmentation_Training_Dataset/train/Segmentation/ | wc -l)"
!echo "Val images:   $(ls {PROJECT_DIR}/Offroad_Segmentation_Training_Dataset/val/Color_Images/ | wc -l)"
!echo "Val masks:    $(ls {PROJECT_DIR}/Offroad_Segmentation_Training_Dataset/val/Segmentation/ | wc -l)"

---
## 📦 Step 3: Install Dependencies

In [ ]:
# Colab already has PyTorch + CUDA, just install extra packages
!pip install -q albumentations>=1.3.0 scipy>=1.10.0 pyyaml>=6.0
print("\n✅ All dependencies installed!")

---
## ⚡ Step 4: Verify GPU

In [ ]:
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU:             {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory:      {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
    print("\n✅ GPU is ready! Training will be FAST 🚀")
else:
    print("\n⚠️  NO GPU DETECTED!")
    print("Go to: Runtime → Change runtime type → GPU (T4)")
    print("Then restart this notebook.")

> **⚠️ If you see 'NO GPU DETECTED':**
> 1. Go to **Runtime** → **Change runtime type**
> 2. Under **Hardware accelerator**, select **T4 GPU**
> 3. Click **Save** and re-run from the beginning

---
## 🏋️ Step 5: Train the Model

This will train DeepLabV3+ with **ResNet101** backbone for **25 epochs**.

Expected time on T4 GPU: **~30-45 minutes**

In [ ]:
%cd /content/offroad_segmentation

# Train with full power settings (config.py auto-detects CUDA)
# ResNet101 backbone, batch_size=8, 2 workers
!python train.py --epochs 25

### 🔄 Optional: Resume Training (if disconnected)
If Colab disconnects, re-run Steps 1-3, then use:

In [ ]:
# Uncomment and run this ONLY if you need to resume after a disconnect
# %cd /content/offroad_segmentation
# !python train.py --epochs 25 --resume checkpoints/best_model.pth

---
## 📊 Step 6: Visualize Training Results

In [ ]:
%cd /content/offroad_segmentation
!python visualize.py

# Display the training curves inline
from IPython.display import Image, display
import os

curves_path = "outputs/training_curves.png"
if os.path.exists(curves_path):
    display(Image(filename=curves_path, width=800))

loss_path = "outputs/loss_curve.png"
if os.path.exists(loss_path):
    display(Image(filename=loss_path, width=600))

---
## 🧪 Step 7: Evaluate on Validation Set

In [ ]:
%cd /content/offroad_segmentation

# Run evaluation with Test-Time Augmentation for best IoU
!python test.py --tta --num_vis 15

In [ ]:
# Display confusion matrix
from IPython.display import Image, display
import os

cm_path = "predictions/confusion_matrix.png"
if os.path.exists(cm_path):
    display(Image(filename=cm_path, width=600))

# Display some comparison images
comp_dir = "predictions/comparisons"
if os.path.exists(comp_dir):
    for f in sorted(os.listdir(comp_dir))[:5]:
        print(f"\n--- {f} ---")
        display(Image(filename=os.path.join(comp_dir, f), width=800))

---
## 💾 Step 8: Save Results to Google Drive

Save the trained model and outputs back to Google Drive so you can download them to your Mac.

In [ ]:
import shutil
import os

# Save destination on Google Drive
SAVE_DIR = os.path.join(DRIVE_FOLDER, "offroad_results")
os.makedirs(SAVE_DIR, exist_ok=True)

# Copy the trained model
best_model = "/content/offroad_segmentation/checkpoints/best_model.pth"
if os.path.exists(best_model):
    shutil.copy2(best_model, os.path.join(SAVE_DIR, "best_model.pth"))
    size_mb = os.path.getsize(best_model) / (1024 * 1024)
    print(f"✅ Model saved ({size_mb:.1f} MB)")

# Copy training outputs
outputs_src = "/content/offroad_segmentation/outputs"
outputs_dst = os.path.join(SAVE_DIR, "outputs")
if os.path.exists(outputs_src):
    shutil.copytree(outputs_src, outputs_dst, dirs_exist_ok=True)
    print("✅ Training outputs saved (curves, history, results)")

# Copy prediction visualizations
preds_src = "/content/offroad_segmentation/predictions"
preds_dst = os.path.join(SAVE_DIR, "predictions")
if os.path.exists(preds_src):
    shutil.copytree(preds_src, preds_dst, dirs_exist_ok=True)
    print("✅ Predictions saved (masks, comparisons, confusion matrix)")

print(f"\n📂 All results saved to: {SAVE_DIR}")
print("\nYou can now download from Google Drive to your Mac!")

---
## 📥 Step 9: Download Model Directly (Alternative)

If you prefer to download the model file directly from Colab:

In [ ]:
# Option A: Download the model directly from Colab
from google.colab import files

# Download the best model
if os.path.exists("/content/offroad_segmentation/checkpoints/best_model.pth"):
    files.download("/content/offroad_segmentation/checkpoints/best_model.pth")
    print("✅ Model download started!")

# Download training results
if os.path.exists("/content/offroad_segmentation/outputs/final_results.json"):
    files.download("/content/offroad_segmentation/outputs/final_results.json")
    print("✅ Results download started!")

---
## 🖥️ Step 10: Use the Model on Your Mac

After downloading `best_model.pth` to your Mac:

```bash
# Copy the model to your project
cp ~/Downloads/best_model.pth /Users/abhimanraj/Downloads/Offroad_Segmentation_Scripts/ENV_SETUP/checkpoints/

# Activate environment
cd /Users/abhimanraj/Downloads/Offroad_Segmentation_Scripts/ENV_SETUP
source EDU_env/bin/activate

# Run evaluation locally
python test.py --tta
```

The model trained on Colab with ResNet101 will work on your Mac's MPS/CPU for inference!

---

## 📋 Troubleshooting

| Issue | Solution |
|-------|----------|
| `No GPU` | Runtime → Change runtime type → T4 GPU |
| `Out of memory` | Reduce batch size: `!python train.py --batch 4` |
| `Disconnected` | Re-run Steps 1-3, then resume: `--resume checkpoints/best_model.pth` |
| `Zip not found` | Make sure files are uploaded to Google Drive root (`My Drive/`) |
| `Slow training` | Check GPU: `!nvidia-smi` — should show T4 or better |